In [ ]:
import os, sys, json, subprocess
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

DATA_DIR = Path("data")
csv_paths = sorted(DATA_DIR.rglob("*.csv"))
print(f"Found {len(csv_paths)} CSV files under {DATA_DIR.resolve()}")

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

OPENROUTER_API_KEY = os.getenv("API_10_KEY")

MODEL = os.getenv("SELECT_MODEL")

def _make_llm():
    return ChatOpenAI(
        model=MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        temperature=0,
        timeout=60,
        max_retries=2,
    )

# (оставляем для совместимости с остальными ячейками)
llm = _make_llm()

In [3]:
MODEL

'nvidia/nemotron-3-nano-30b-a3b:free'

In [4]:
PROMPT_RU = r"""
Тебе дана таблица в формате markdown.
Используй ТОЛЬКО данные из этой таблицы.

Твоя задача — сгенерировать 7 сложных вопросов для оценки RAG-системы.

ТРЕБОВАНИЯ К ВОПРОСАМ:

1. Вопрос должен требовать анализа таблицы, а не простого копирования одной ячейки.
2. Вопрос НЕ должен быть решаем без доступа к таблице.
3. Не используй общеизвестные факты или знания вне таблицы.
4. Минимум 5 из 7 вопросов должны требовать:
   - фильтрации по условию
   - сравнения значений
   - поиска максимума/минимума
   - подсчёта (COUNT)
   - работы с несколькими строками
5. Избегай вопросов типа:
   - "Какой X у Y?"
   - прямого поиска одного значения
   - вопросов, где ответ — просто уникальная ячейка

6. Если таблица не позволяет создать сложный вопрос — не придумывай информацию.
7. Каждый вопрос ОБЯЗАТЕЛЬНО должен явно содержать название статьи.
8. Если у таблицы есть название — оно также должно быть явно упомянуто в вопросе.

ВАЖНО ДЛЯ ПОСЛЕДУЮЩЕГО ПОИСКА (ретривер):
Каждый вопрос должен быть самодостаточным по одному тексту question.
Поэтому в каждом question ЕСТЕСТВЕННО (без тегов и меток) упоминай:
  - название статьи
  - название таблицы
Эти названия бери из блока «МЕТАДАННЫЕ» в сообщении пользователя.

ТРЕБОВАНИЯ К ИНТЕГРАЦИИ МЕТЫ:
 - Вопрос должен звучать как обычный человеческий вопрос (естественный русский).
 - НЕ используй явные шаблоны/шапки типа: "[ARTICLE: ...]", "TABLE:" и т.п.
 - Можно упоминать названия в кавычках/скобках, например:
   "…в таблице «Результаты» по теме «Парламентские выборы в Норвегии (1894)»…"
 - МЕТАДАННЫЕ — не источник фактов; факты/ответ вычисляй только из таблицы.

ДЛЯ КАЖДОГО ВОПРОСА ВЕРНИ:

- question (на русском, естественный, и содержит названия статьи и таблицы)
- answer (точный ответ, как строка или список строк)
- reasoning (краткое объяснение, как получен ответ)
- supporting_rows (список индексов строк, которые использовались)
- supporting_columns — список названий столбцов, использованных при ответе
- question_type (один из: filter, aggregation, comparison, argmax, argmin, count, multi_condition)

Формат вывода — строго JSON:

[
  {
    "question": "...",
    "answer": "...",
    "reasoning": "...",
    "supporting_rows": [ ... ],
    "supporting_columns": [ ... ],
    "question_type": "..."
  }
]

Если вопрос можно решить без таблицы — НЕ создавай его.
Не добавляй информацию, которой нет в таблице.
Ответ должен быть полностью вычислим из таблицы.
""".strip()

SYSTEM = "Ты аккуратно работаешь с табличными данными и возвращаешь строго валидный JSON без лишнего текста."

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _extract_json_array(text: str) -> str:
    text = (text or "").strip()
    s = text.find("[")
    e = text.rfind("]")
    if s != -1 and e != -1 and e > s:
        return text[s:e+1]
    return text

def _is_ready_json(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False
    if not isinstance(obj, list) or len(obj) == 0:
        return False
    return all(isinstance(x, dict) and "question" in x and "answer" in x for x in obj)

OUT_DIR = Path("rag_questions")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_READY_JSON = int(os.environ.get("MAX_READY_JSON", "5000"))
MIN_TABLE_ROWS = int(os.environ.get("MIN_TABLE_ROWS", "3"))
MAX_WORKERS = int(os.environ.get("MAX_WORKERS", "8"))

def _paths_for(csv_path: Path):
    try:
        rel = csv_path.relative_to(DATA_DIR)
    except Exception:
        rel = Path(csv_path.name)
    out_path = (OUT_DIR / rel).with_suffix(".json")
    err_path = out_path.with_suffix(".error.json")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    return rel, out_path, err_path

def process_one(csv_path: Path) -> dict:
    rel, out_path, err_path = _paths_for(csv_path)

    # resume (учитываем и out_path, и err_path)
    if _is_ready_json(out_path):
        return {"status": "skip_ready", "rel": str(rel), "out": str(out_path)}
    if err_path.exists():
        return {"status": "skip_error_logged", "rel": str(rel), "out": str(out_path)}

    # read
    try:
        df = pd.read_csv(csv_path, sep="|", index_col=0)
    except Exception as e:
        err_path.write_text(json.dumps({"table_file": str(csv_path), "stage": "read_csv", "error": str(e)}, ensure_ascii=False, indent=2), encoding="utf-8")
        return {"status": "error", "stage": "read_csv", "rel": str(rel), "out": str(out_path)}

    # skip small tables
    if df.shape[0] < MIN_TABLE_ROWS:
        return {"status": "skip_small", "rows": int(df.shape[0]), "rel": str(rel), "out": str(out_path)}

    table_md = df.to_markdown()

    # --- meta
    meta_path = csv_path.with_name(csv_path.stem + "_meta.json")
    article_title = None
    table_title = None
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            article_title = (meta.get("page") or {}).get("title")
            table_title = meta.get("title")
        except Exception:
            pass

    article_title_s = article_title or "unknown"
    table_title_s = table_title or "unknown"

    if article_title_s is None:
        print(meta_path)

    meta_block = (
        "МЕТАДАННЫЕ (только для ориентира; НЕ использовать как источник фактов):\n"
        f"Название статьи: {article_title_s}\n"
        f"Название таблицы: {table_title_s}\n"
    )

    user_content = (
        PROMPT_RU
        + "\n\n"
        + meta_block
        + "\n"
        + "ТАБЛИЦА (markdown):\n\n"
        + table_md
    )

    messages = [
        SystemMessage(content=SYSTEM),
        HumanMessage(content=user_content),
    ]

    # call
    try:
        local_llm = _make_llm()  # отдельный клиент на поток
        resp = local_llm.invoke(messages)
    except Exception as e:
        err_path.write_text(json.dumps({"table_file": str(csv_path), "stage": "openrouter", "error": str(e)}, ensure_ascii=False, indent=2), encoding="utf-8")
        return {"status": "error", "stage": "openrouter", "rel": str(rel), "out": str(out_path)}

    raw = getattr(resp, "content", "")
    candidate = _extract_json_array(raw)

    # parse+write
    try:
        parsed = json.loads(candidate)
        if not (isinstance(parsed, list) and all(isinstance(x, dict) for x in parsed)):
            raise ValueError("model_output_is_not_json_array_of_objects")

        # Страховка: если модель не вставила названия в question — добавляем незаметно и естественно.
        # Это важно, чтобы по одному вопросу можно было ретривить нужную таблицу.
        natural_prefix = f"По таблице «{table_title_s}» ({article_title_s}) "
        for item in parsed:
            q = item.get("question")
            if not isinstance(q, str) or not q.strip():
                continue
            q_str = q.strip()
            # проверяем наличие подстрок как минимум по article_title/table_title
            has_article = (article_title_s != "unknown") and (article_title_s in q_str)
            has_table = (table_title_s != "unknown") and (table_title_s in q_str)
            if (article_title_s == "unknown" or has_article) and (table_title_s == "unknown" or has_table):
                continue
            if not q_str.startswith(natural_prefix):
                item["question"] = natural_prefix + q_str

        out_path.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
        if err_path.exists():
            err_path.unlink()
        return {"status": "ok", "rel": str(rel), "out": str(out_path)}
    except Exception as e:
        err_path.write_text(json.dumps({"table_file": str(csv_path), "stage": "json_parse", "error": str(e), "raw": raw}, ensure_ascii=False, indent=2), encoding="utf-8")
        return {"status": "error", "stage": "json_parse", "rel": str(rel), "out": str(out_path)}

# --- build candidate list + initial ready_count
ready_count = 0
candidates = []
for p in csv_paths:
    _, out_path, err_path = _paths_for(p)
    if _is_ready_json(out_path):
        ready_count += 1
    elif err_path.exists():
        pass
    else:
        candidates.append(p)

print(f"Ready existing: {ready_count} / {MAX_READY_JSON}; candidates: {len(candidates)}; workers: {MAX_WORKERS}")
target_new = max(0, MAX_READY_JSON - ready_count)

submitted = 0
completed = 0
futures = set()
idx = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    pbar = tqdm(total=target_new, desc="ready json", unit="file")

    def submit_until_full():
        global idx, submitted
        while idx < len(candidates) and len(futures) < MAX_WORKERS and ready_count < MAX_READY_JSON:
            futures.add(ex.submit(process_one, candidates[idx]))
            idx += 1
            submitted += 1

    submit_until_full()

    while futures:
        for fut in as_completed(futures):
            futures.remove(fut)
            completed += 1
            try:
                res = fut.result()
            except Exception as e:
                res = {"status": "error", "stage": "executor", "error": str(e)}

            if res.get("status") == "ok":
                ready_count += 1
                pbar.update(1)
                print(f"WRITE: {res.get('rel')} -> {res.get('out')}")
                pbar.set_postfix_str(f"ready={ready_count}/{MAX_READY_JSON}")
                if ready_count >= MAX_READY_JSON:
                    for f in list(futures):
                        f.cancel()
                    futures = {f for f in futures if not f.cancelled()}
            
            if res.get("status") == 'error':
                print(res)

            submit_until_full()
            break

    pbar.close()

print(f"Ready JSONs (counting existing + new): {ready_count} / {MAX_READY_JSON}")
print(f"Submitted: {submitted}, completed: {completed}, remaining candidates not submitted: {max(0, len(candidates)-idx)}")

In [ ]:
for error_path in Path("rag_questions").rglob("*.error.json"):
    print(error_path)
    error_file = json.loads(error_path.read_text(encoding="utf-8"))
    if "name 'PROMPT_RU' is not define" in error_file.get("error", ""):
        error_path.unlink()

In [ ]:
from pathlib import Path
import json
from collections import Counter

BASE = Path("rag_questions")
ready_paths = sorted([p for p in BASE.rglob("*.json") if not p.name.endswith(".error.json")])
error_paths = sorted(BASE.rglob("*.error.json"))

print(f"ready json files: {len(ready_paths)}")
print(f"error json files: {len(error_paths)}")

total_questions = 0
question_type_ctr = Counter()
missing_keys_ctr = Counter()
supporting_cols_ctr = Counter()
supporting_rows_len_ctr = Counter()
bad_ready_files = 0

REQUIRED_KEYS = {"question", "answer", "reasoning", "supporting_rows", "supporting_columns", "question_type"}

for p in ready_paths:
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
        if not isinstance(obj, list):
            raise ValueError("not_a_list")
    except Exception:
        bad_ready_files += 1
        continue

    total_questions += len(obj)
    for item in obj:
        if not isinstance(item, dict):
            missing_keys_ctr["__non_dict_item__"] += 1
            continue

        missing = REQUIRED_KEYS - set(item.keys())
        for k in missing:
            missing_keys_ctr[k] += 1

        qt = item.get("question_type")
        if isinstance(qt, str) and qt:
            question_type_ctr[qt] += 1
        else:
            question_type_ctr["__missing_question_type__"] += 1

        cols = item.get("supporting_columns")
        if isinstance(cols, list):
            for c in cols:
                if isinstance(c, str) and c:
                    supporting_cols_ctr[c] += 1
        else:
            supporting_cols_ctr["__missing_supporting_columns__"] += 1

        srows = item.get("supporting_rows")
        if isinstance(srows, list):
            supporting_rows_len_ctr[len(srows)] += 1
        else:
            supporting_rows_len_ctr[-1] += 1

avg_q_per_table = (total_questions / len(ready_paths)) if ready_paths else 0

print("\n--- questions stats ---")
print(f"total questions: {total_questions}")
print(f"avg questions per ready file: {avg_q_per_table:.2f}")
print(f"bad ready files (json unreadable / not list): {bad_ready_files}")

print("\nquestion_type distribution:")
for k, v in question_type_ctr.most_common():
    print(f"  {k}: {v}")

print("\nmissing keys counts (across items):")
for k, v in missing_keys_ctr.most_common():
    print(f"  {k}: {v}")

print("\nsupporting_rows length distribution:")
for k, v in sorted(supporting_rows_len_ctr.items(), key=lambda x: x[0]):
    label = "__missing__" if k == -1 else str(k)
    print(f"  {label}: {v}")

print("\ntop supporting_columns:")
for k, v in supporting_cols_ctr.most_common(30):
    print(f"  {k}: {v}")

# --- error stats
stage_ctr = Counter()
for p in error_paths:
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
        st = obj.get("stage") if isinstance(obj, dict) else None
        stage_ctr[st or "__missing_stage__"] += 1
    except Exception:
        stage_ctr["__unreadable_error_json__"] += 1

print("\n--- error stats ---")
for k, v in stage_ctr.most_common():
    print(f"  {k}: {v}")

In [3]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
import json, os, random, zipfile
import pandas as pd
from IPython.display import display, Markdown

@dataclass(frozen=True)
class QAItem:
    json_path: Path
    rel_json: Path
    csv_path: Path
    q_idx: int
    q: dict

def load_question_bank(base_dir: str | Path = "rag_questions", data_dir: str | Path = "data") -> list[QAItem]:
    base_dir = Path(base_dir)
    data_dir = Path(data_dir)
    ready_paths = sorted([p for p in base_dir.rglob("*.json") if not p.name.endswith(".error.json")])
    items: list[QAItem] = []
    bad_files = 0

    for jp in ready_paths:
        try:
            obj = json.loads(jp.read_text(encoding="utf-8"))
            if not isinstance(obj, list):
                raise ValueError("root_not_list")
        except Exception:
            bad_files += 1
            continue

        rel_json = jp.relative_to(base_dir)  # e.g. 8978114/table_0.json
        csvp = (data_dir / rel_json).with_suffix(".csv")

        for q_idx, q in enumerate(obj):
            if isinstance(q, dict):
                items.append(QAItem(json_path=jp, rel_json=rel_json, csv_path=csvp, q_idx=q_idx, q=q))

    items.sort(key=lambda x: (str(x.rel_json), x.q_idx))
    print(f"ready json files: {len(ready_paths)}; total questions: {len(items)}; bad json files skipped: {bad_files}")
    return items

def select_items(
    items: list[QAItem],
    mode: str,
    *,
    n: int = 100,
    start: int = 1,
    index: int = 100,
    seed: int | None = None,
) -> list[QAItem]:
    if not items:
        return []
    mode = (mode or "").lower()

    if mode == "index":
        idx0 = max(0, min(len(items) - 1, index - 1))
        return [items[idx0]]

    if mode == "random":
        rng = random.SystemRandom() if seed is None else random.Random(seed)
        if seed is not None:
            print(f"random seed={seed}")
        total = len(items)
        if n >= total:
            # если просим весь набор — будет тот же набор, меняется только порядок
            items2 = items[:]
            rng.shuffle(items2)
            print(f"[random] n={n} >= total={total}: returning ALL questions in random order")
            return items2
        return rng.sample(items, k=n)

    # default: slice (1-based)
    start0 = max(0, start - 1)
    end0 = min(len(items), start0 + n)
    return items[start0:end0]

def df_to_md(df: pd.DataFrame, *, table_max_rows: int = 30) -> str:
    n = len(df)
    if n <= table_max_rows:
        return df.to_markdown()
    half = max(1, table_max_rows // 2)
    head = df.head(half)
    tail = df.tail(table_max_rows - half)
    return head.to_markdown() + "\n\n...\n\n" + tail.to_markdown()

def export_items_to_markdown(
    items: list[QAItem],
    *,
    out_dir: str | Path = "review_outputs",
    basename: str = "review",
    table_max_rows: int = 30,
) -> Path | None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_md = out_dir / f"{basename}_{ts}.md"

    md = []
    md.append(f"# RAG questions review\n\nGenerated: {datetime.now().isoformat()}\n")
    md.append(f"Total items: {len(items)}\n")

    for i, it in enumerate(items, 1):
        q = it.q or {}
        md.append("\n---\n")
        md.append(f"## [{i}/{len(items)}] {it.rel_json} — вопрос #{it.q_idx + 1}\n")
        md.append(f"**json:** `{it.json_path}`  ")
        md.append(f"**csv:** `{it.csv_path}`  ")
        md.append(f"**question_type:** `{q.get('question_type')}`  ")
        md.append(f"**supporting_rows:** `{q.get('supporting_rows')}`  ")
        md.append(f"**supporting_columns:** `{q.get('supporting_columns')}`\n")

        md.append("### Вопрос\n")
        md.append((q.get("question") or "").strip() + "\n")

        md.append("\n### Ответ\n")
        md.append("```json\n" + json.dumps(q.get("answer"), ensure_ascii=False, indent=2) + "\n```\n")

        md.append("\n### Reasoning\n")
        md.append((q.get("reasoning") or "").strip() + "\n")

        md.append("\n### Таблица (markdown)\n")
        if not it.csv_path.exists():
            md.append("**<CSV не найден>**\n")
            continue
        try:
            df = pd.read_csv(it.csv_path, sep="|", index_col=0)
            md.append(df_to_md(df, table_max_rows=table_max_rows) + "\n")
        except Exception as e:
            md.append(f"**<ошибка чтения CSV: {e}>**\n")

    out_md.write_text("\n".join(md), encoding="utf-8")

    return out_md

def render_item(it: QAItem, *, i: int | None = None, total: int | None = None, table_max_rows: int = 30):
    q = it.q
    prefix = f"[{i}/{total}] " if (i is not None and total is not None) else ""
    title = f"## {prefix}{it.rel_json} — вопрос #{it.q_idx + 1}"
    meta = (
        f"**json:** `{it.json_path}`  \n"
        f"**csv:** `{it.csv_path}`  \n"
        f"**question_type:** `{q.get('question_type')}`  \n"
        f"**supporting_rows:** `{q.get('supporting_rows')}`  \n"
        f"**supporting_columns:** `{q.get('supporting_columns')}`"
    )
    body = (
        f"### Вопрос\n{q.get('question')}\n\n"
        f"### Ответ\n```json\n{json.dumps(q.get('answer'), ensure_ascii=False, indent=2)}\n```\n\n"
        f"### Reasoning\n{q.get('reasoning')}\n"
    )

    display(Markdown(title))
    display(Markdown(meta))
    display(Markdown(body))

    if not it.csv_path.exists():
        display(Markdown("### Таблица\n**<CSV не найден>**"))
        return
    try:
        df = pd.read_csv(it.csv_path, sep="|", index_col=0)
    except Exception as e:
        display(Markdown(f"### Таблица\n**<ошибка чтения CSV: {e}>**"))
        return

    display(Markdown("### Таблица (markdown)"))
    display(Markdown(df_to_md(df, table_max_rows=table_max_rows)))

def show_items(items: list[QAItem], *, table_max_rows: int = 30):
    total = len(items)
    for i, it in enumerate(items, 1):
        render_item(it, i=i, total=total, table_max_rows=table_max_rows)

In [ ]:
# Подход 1: сохраняем в файл (без вывода в output)
bank = load_question_bank()
picked = select_items(bank, mode="slice", n=200, start=1)
out_md, out_zip = export_items_to_markdown(picked, basename="slice_200", table_max_rows=30, zip_output=True)
print("WROTE:", out_md.resolve())
print("ZIPPED:", out_zip.resolve() if out_zip else None)

In [ ]:
# Подход 2: сохраняем в файл (без вывода в output)
bank = load_question_bank()
picked = select_items(bank, mode="random", n=100)  # seed=None => SystemRandom
out_md = export_items_to_markdown(picked, basename="random_100", table_max_rows=1000)
print("WROTE:", out_md.resolve())

In [ ]:
# Подсчёт количества символов в markdown-представлении каждой таблицы из csv_paths

records = []

for p in tqdm(csv_paths, desc="Counting markdown chars"):
    try:
        df = pd.read_csv(p, sep="|", index_col=0)
        md = df.to_markdown()
        records.append({
            "csv_path": str(p),
            "rows": int(df.shape[0]),
            "cols": int(df.shape[1]),
            "markdown_chars": len(md),
        })
    except Exception as e:
        records.append({
            "csv_path": str(p),
            "rows": None,
            "cols": None,
            "markdown_chars": None,
            "error": str(e),
        })

md_char_stats = pd.DataFrame(records)

# Успешно обработанные таблицы
ok = md_char_stats[md_char_stats["markdown_chars"].notna()].copy()
ok = ok.sort_values("markdown_chars", ascending=False)

print(f"Всего CSV: {len(md_char_stats)}")
print(f"Успешно обработано: {len(ok)}")
print(f"С ошибками: {len(md_char_stats) - len(ok)}")

if len(ok) > 0:
    print("\n--- Сводка по символам markdown ---")
    print(f"Сумма символов: {int(ok['markdown_chars'].sum())}")
    print(f"Среднее: {ok['markdown_chars'].mean():.2f}")
    print(f"Медиана: {ok['markdown_chars'].median():.2f}")
    print(f"Минимум: {int(ok['markdown_chars']).min()}")
    print(f"Максимум: {int(ok['markdown_chars']).max()}")

    print("\nТоп-20 самых длинных таблиц (по markdown_chars):")
    display(ok.head(20))

# Если хотите, можно сохранить результат:
# ok.to_csv("markdown_char_stats.csv", index=False)